# Togather — Analysis

**Goal:** Identify growth opportunities and commercial recommendations for 2025.  
**Builds on:** `01_eda.ipynb` — run that first to understand data quality issues.

## 0. Setup

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

FIGURES = '../outputs/figures/'
RAW     = '../data/raw/'

In [ ]:
def load_requests(path=RAW + 'food_requests.xlsx'):
    df = pd.read_excel(path, header=1)
    df.columns = ['event_request_id', 'created', 'region', 'priority_tag', 'request_budget']
    df['created'] = pd.to_datetime(df['created'], format='mixed')
    df['priority_tag'] = df['priority_tag'].str.strip()
    return df

def load_quotes(path=RAW + 'food_quotes.xlsx'):
    df = pd.read_excel(path)
    df.columns = [
        'quote_id', 'supplier_id', 'event_request_id',
        'quote_created', 'booked', 'supplier_region',
        'quote_price', 'supplier_primary_tag'
    ]
    df['quote_created'] = pd.to_datetime(df['quote_created'], format='mixed')
    df['booked'] = df['booked'].astype(int)
    df['supplier_primary_tag'] = df['supplier_primary_tag'].str.strip()
    return df

requests = load_requests()
quotes   = load_quotes()
print('Requests:', requests.shape)
print('Quotes:  ', quotes.shape)

---
## 1. Macro Category Mapping

In [ ]:
TAG_GROUPS = {
    # Street Food
    'Pizza':          'Street Food',
    'Burgers':        'Street Food',
    'Hot Dogs':       'Street Food',
    'Fries':          'Street Food',
    'Wraps':          'Street Food',
    'Sandwiches':     'Street Food',
    'Fish and Chips': 'Street Food',
    'Fried Chicken':  'Street Food',
    'Mac & Cheese':   'Street Food',
    # BBQ & Meats
    'BBQ':          'BBQ & Meats',
    'Hog Roast':    'BBQ & Meats',
    'Pulled Meats': 'BBQ & Meats',
    'Pies':         'BBQ & Meats',
    # Asian & Indian
    'Asian':      'Asian & Indian',
    'Chinese':    'Asian & Indian',
    'Indian':     'Asian & Indian',
    'Japanese':   'Asian & Indian',
    'Thai':       'Asian & Indian',
    'Vietnamese': 'Asian & Indian',
    'Korean':     'Asian & Indian',
    'Curries':    'Asian & Indian',
    # World & Fusion
    'Mexican':        'World & Fusion',
    'Caribbean':      'World & Fusion',
    'South American': 'World & Fusion',
    'African':        'World & Fusion',
    'Middle Eastern': 'World & Fusion',
    'Fusion':         'World & Fusion',
    'Seafood':        'World & Fusion',
    # European
    'Italian':        'European',
    'French':         'European',
    'Greek':          'European',
    'Spanish':        'European',
    'Mediterranean':  'European',
    'Pasta':          'European',
    'Modern British': 'European',
    'Fine dining':    'European',
    # Sweet & Desserts
    'Dessert':             'Sweet & Desserts',
    'Ice Cream':           'Sweet & Desserts',
    'Waffles':             'Sweet & Desserts',
    'Doughnuts & Churros': 'Sweet & Desserts',
    'Cr\u00eapes':         'Sweet & Desserts',
    'Afternoon Tea':       'Sweet & Desserts',
    'Wedding Cake':        'Sweet & Desserts',
    # Sharing & Events
    'Canap\u00e9s':             'Sharing & Events',
    'Salads':                   'Sharing & Events',
    'Grazing boards or tables': 'Sharing & Events',
    'Gifting & Hampers':        'Sharing & Events',
    'Breakfast':                'Sharing & Events',
    'Seasonal':                 'Sharing & Events',
    'Festive':                  'Sharing & Events',
    # Dietary
    'Strictly Vegan': 'Dietary',
    'Vegan':          'Dietary',
    'Sustainable':    'Dietary',
    # Bar & Drinks
    'Indoor bar':         'Bar & Drinks',
    'Horsebox bar':       'Bar & Drinks',
    'Oh-wow outdoor bar': 'Bar & Drinks',
    'Outdoors':           'Bar & Drinks',
    'Rooftop':            'Bar & Drinks',
    'Beer & Cider':       'Bar & Drinks',
    # Service
    'Waiters':    'Service',
    'Bartenders': 'Service',
    'Vehicle':    'Service',
    'Delivery':   'Service',
}

def apply_macro(tag):
    if pd.isna(tag):
        return None
    return TAG_GROUPS.get(str(tag).strip(), 'Other')

requests['macro_category'] = requests['priority_tag'].map(apply_macro)
quotes['macro_category']   = quotes['supplier_primary_tag'].map(apply_macro)

# Coverage check
for name, df in [('requests', requests), ('quotes', quotes)]:
    mapped = df['macro_category'].notna() & (df['macro_category'] != 'Other')
    other  = (df['macro_category'] == 'Other').sum()
    null   = df['macro_category'].isna().sum()
    print(f'{name}: mapped={mapped.sum():,} ({mapped.mean():.1%}) | other={other:,} | null={null:,}')

print()
print('Unmapped (requests):', requests[requests['macro_category']=='Other']['priority_tag'].unique().tolist())
print('Unmapped (quotes):  ', quotes[quotes['macro_category']=='Other']['supplier_primary_tag'].unique().tolist())

---
## 2. Base Merge

One row per quote (requests with no quotes appear once with NaN quote fields).

In [ ]:
df = requests.merge(
    quotes.rename(columns={'macro_category': 'supplier_macro'}),
    on='event_request_id',
    how='left'
).rename(columns={'macro_category': 'customer_macro'})

# Convenience flags
df['has_quote']  = df['quote_id'].notna()
df['is_booked']  = df['booked'].fillna(0).astype(int)
df['month']      = df['created'].dt.to_period('M')
df['quarter']    = df['created'].dt.to_period('Q')

print('Merged shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(3)